# Attach text column to existing joint-classifier predictions

The chapter 4.4.5 training notebook saved test-set predictions in a file
with columns `y_true, y_pred, true_label, pred_label, prob_0, ..., prob_10`,  but **without the input text**. The pipeline-comparison notebook (03)
joins the three sources of predictions on the `text` column, so the missing
text has to be reattached from the test split CSV.

This notebook does just that. It assumes that the predictions CSV preserves
the row order of the test split it was generated from (the default for
`trainer.predict` in HuggingFace Trainer). A safety check compares the
`true_label` column against the test split's `pattern` column row-by-row;
if they disagree anywhere, the script aborts rather than silently
producing wrong matches.


## 0. Paths and configuration

In [4]:
from pathlib import Path
import pandas as pd

PREDICTIONS_CSV = Path(
    "../02_04_roberta_patternreframe_none_aug/results/02_04_0.3_test_predictions.csv"
)
TEST_CSV = Path("../../processed/patternreframe_test_aug_v1_balanced.csv")
OUT_PATH = Path("results/joint11_patternreframe_test_predictions.csv")

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"PREDICTIONS_CSV: {PREDICTIONS_CSV}")
print(f"TEST_CSV:        {TEST_CSV}")
print(f"OUT_PATH:        {OUT_PATH}")


PREDICTIONS_CSV: ../02_04_roberta_patternreframe_none_aug/results/02_04_0.3_test_predictions.csv
TEST_CSV:        ../../processed/patternreframe_test_aug_v1_balanced.csv
OUT_PATH:        results/joint11_patternreframe_test_predictions.csv


## 1. Load both files

In [5]:
preds = pd.read_csv(PREDICTIONS_CSV)
test = pd.read_csv(TEST_CSV)

print(f"Predictions: {len(preds)} rows, columns: {list(preds.columns)}")
print(f"Test split:  {len(test)} rows, columns: {list(test.columns)}")


Predictions: 7488 rows, columns: ['y_true', 'y_pred', 'true_label', 'pred_label', 'prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4', 'prob_5', 'prob_6', 'prob_7', 'prob_8', 'prob_9', 'prob_10']
Test split:  7488 rows, columns: ['persona', 'pattern', 'pattern_def', 'thought', 'thought_length', 'n_reframes', 'reframe_1', 'reframe_2', 'reframe_3', 'marked_catastrophizing', 'marked_discounting_positive', 'marked_labeling', 'marked_mental_filtering', 'marked_mind_reading', 'marked_fortune_telling', 'marked_overgeneralization', 'marked_personalization', 'marked_black_and_white', 'marked_should_statements', 'marked_emotional_reasoning', 'marked_none', 'strategies_per_reframe']


## 2. Sanity-check row counts and order

The row count must match exactly. The `true_label` column in the predictions
CSV must agree row-by-row with the `pattern` column in the test split, if
not, the row order has drifted somewhere and joining by index would silently
produce wrong matches.


In [6]:
assert len(preds) == len(test), (
    f"Row count mismatch: predictions {len(preds)} vs test {len(test)}. "
    "Re-run inference instead of attaching by index."
)

mismatches = (preds["true_label"].values != test["pattern"].values).sum()
print(f"Row-by-row label mismatches: {mismatches}")
assert mismatches == 0, (
    "Predictions and test split are not in the same order. "
    "Re-run inference on the joint model checkpoint instead."
)
print("OK — row counts and order match.")


Row-by-row label mismatches: 0
OK — row counts and order match.


## 3. Build the joined predictions DataFrame

The output uses the column names the pipeline notebook expects:
`text, gold_label, joint_pred_label`. The label values stay in their original
long form (e.g. `"Jumping to conclusions: mind reading"`), as that is what
the rest of the pipeline uses too.


In [9]:
# Diagnostic: find duplicate texts with conflicting gold labels
diag = pd.DataFrame({
    "text":       test["thought"].astype(str).values,
    "pattern":    test["pattern"].values,
    "pred_label": preds["pred_label"].values,
})
dup_mask = diag["text"].duplicated(keep=False)
dups = diag[dup_mask].sort_values("text")
print(f"All duplicate rows: {len(dups)}")
print(f"Unique duplicate texts: {dups['text'].nunique()}")

# Group by text and inspect each group
for text, grp in dups.groupby("text"):
    labels = grp["pattern"].tolist()
    if len(set(labels)) > 1:
        print(f"\n--- INCONSISTENT ---")
    else:
        print(f"\n--- consistent ---")
    print(f"  text:    {repr(text)}")
    print(f"  labels:  {labels}")
    print(f"  model:   {grp['pred_label'].tolist()}")

All duplicate rows: 11
Unique duplicate texts: 5

--- INCONSISTENT ---
  text:    'I run marathons twice a year and I never won once. I am such a loser'
  labels:  ['Mental filtering', 'Jumping to conclusions: mind reading']
  model:   ['Labeling and mislabeling', 'Labeling and mislabeling']

--- INCONSISTENT ---
  text:    'My cat died. I am such a bad pet owner'
  labels:  ['Jumping to conclusions: Fortune-telling', 'Jumping to conclusions: mind reading', 'Mental filtering']
  model:   ['Personalization', 'Personalization', 'Personalization']

--- INCONSISTENT ---
  text:    'My girlfriend is mad at me because I am saving up to get some procedures done.'
  labels:  ['Personalization', 'Catastrophizing']
  model:   ['Jumping to conclusions: mind reading', 'Jumping to conclusions: mind reading']

--- INCONSISTENT ---
  text:    'People criticize me because I have children without being married.'
  labels:  ['Catastrophizing', 'Labeling and mislabeling']
  model:   ['Jumping to conclusi

### Handle duplicate texts in the augmented test set.
Investigation of the augmented PatternReframe test split revealed:

(a) Six texts on the No Distortion side stem from Bahasa back-translation collisions (different Indonesian survey responses translating to identical English) and carry consistent gold labels.

(b) Five texts on the PatternReframe positive side appear in multiple records with different 'pattern' fields. Inspection of the underlying JSON shows the marked_patterns annotator distribution is IDENTICAL across these duplicates (e.g., all three records for "My cat died. I am such a bad pet owner" have 4 votes for Labeling and 4 votes for Personalization). The variation lies in the 'pattern' prompt presented to annotators, not in their judgement. This is a known property of the PatternReframe dataset, where each (thought, pattern) prompt was a separate annotation task; the 'pattern' field is the prompt context rather than a dominant annotator-assigned label.

Every published number on PatternReframe treats the 'pattern' field as the gold label, including for duplicate thoughts. To keep the chapter 4.5 numbers directly comparable to the chapter 4.4 numbers, all 11 records
are retained. The merge in notebook 03 disambiguates them by joining on (text, gold_label) jointly, since the (text, gold_label) pair is unique in the test set.

In [10]:
out = pd.DataFrame({
    "text":             test["thought"].astype(str).values,
    "gold_label":       preds["true_label"].values,
    "joint_pred_idx":   preds["y_pred"].values,
    "joint_pred_label": preds["pred_label"].values,
})

prob_cols = [c for c in preds.columns if c.startswith("prob_")]
for c in prob_cols:
    out[f"joint_{c}"] = preds[c].values

n_dup_rows = out["text"].duplicated().sum()
n_dup_texts = out["text"].duplicated().sum() and out.loc[out["text"].duplicated(keep=False), "text"].nunique()
print(f"Test set: {len(out)} rows total.")
print(f"  {n_dup_rows} rows are duplicates of an earlier row (by text alone).")
print(f"  {n_dup_texts} distinct texts have multiple records.")
print(f"All rows retained; notebook 03 merges on (text, gold_label) jointly.")

# Sanity check: (text, gold_label) must be unique. If it isn't, two records
# in the predictions CSV have the same text AND the same gold label, which
# would be a true duplicate (the model predicts the same thing twice for
# the same gold label) and indicates an upstream issue.
key_dup = out.duplicated(subset=["text", "gold_label"]).sum()
assert key_dup == 0, (
    f"{key_dup} rows have the same (text, gold_label) pair. The joint-key "
    "merge in notebook 03 will fail. Inspect manually."
)
print("(text, gold_label) is unique — joint merge will succeed.")

out.head()

Test set: 7488 rows total.
  6 rows are duplicates of an earlier row (by text alone).
  5 distinct texts have multiple records.
All rows retained; notebook 03 merges on (text, gold_label) jointly.
(text, gold_label) is unique — joint merge will succeed.


,text,gold_label,joint_pred_idx,joint_pred_label,joint_prob_0,joint_prob_1,joint_prob_2,joint_prob_3,joint_prob_4,joint_prob_5,joint_prob_6,joint_prob_7,joint_prob_8,joint_prob_9,joint_prob_10
0,People only want to be my friend because I'm p...,Jumping to conclusions: mind reading,4,Jumping to conclusions: mind reading,0.015255,0.015235,0.131416,0.029805,0.577820,0.030816,0.054334,0.020180,0.058473,0.035974,0.030692
1,"I love to paint pictures, but I'll not make to...",Jumping to conclusions: Fortune-telling,8,Overgeneralization,0.063400,0.026510,0.076749,0.301576,0.017933,0.018788,0.049618,0.010749,0.408862,0.010317,0.015499
2,I love to snack between meals. I am going to b...,Jumping to conclusions: Fortune-telling,3,Jumping to conclusions: Fortune-telling,0.064389,0.226483,0.006633,0.437305,0.017640,0.017897,0.017489,0.003230,0.178491,0.023540,0.006903
3,I like watching The Walking Dead. Too bad I do...,Labeling and mislabeling,2,Discounting the positive,0.061915,0.010024,0.541994,0.014545,0.010067,0.035942,0.229505,0.015341,0.042037,0.028569,0.010062
4,I hate two cats. People must think I'm clearly...,Jumping to conclusions: mind reading,4,Jumping to conclusions: mind reading,0.014928,0.026103,0.006652,0.060878,0.769263,0.014271,0.020474,0.004312,0.059203,0.011480,0.012434


## 4. Write the output CSV

In [11]:
out.to_csv(OUT_PATH, index=False)
print(f"Wrote {len(out)} rows to {OUT_PATH}")
print(f"Output columns: {list(out.columns)}")


Wrote 7488 rows to results/joint11_patternreframe_test_predictions.csv
Output columns: ['text', 'gold_label', 'joint_pred_idx', 'joint_pred_label', 'joint_prob_0', 'joint_prob_1', 'joint_prob_2', 'joint_prob_3', 'joint_prob_4', 'joint_prob_5', 'joint_prob_6', 'joint_prob_7', 'joint_prob_8', 'joint_prob_9', 'joint_prob_10']
